[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/02_Vision_Language_Models/02_image_captioning/02_image_captioning.ipynb)

# 02. Image Captioning: Image → Text Generation

**This notebook covers:**
- Image captioning architecture (encoder-decoder)
- Building a small captioning model from scratch
- Using pretrained models (BLIP) for captioning
- Visualizing attention: which image regions generate which words

---

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/02_Vision_Language_Models/02_image_captioning")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from utils.visualization import *
from utils.helpers import count_parameters

set_style()

## 1. Architecture: Encoder-Decoder for Captioning

```
Image → Vision Encoder → Image Features (K, V)
                                    ↓
[BOS] → Text Decoder (Q) ← Cross-Attention → "a" → "cat" → "sitting" → [EOS]
```

The decoder generates one word at a time, attending to image features.

### Autoregressive Generation Math

Image captioning models the conditional probability of a caption $\mathbf{y} = (y_1, \ldots, y_T)$ given image $I$ as a **product of conditionals** (chain rule):

$$P(\mathbf{y} \mid I) = \prod_{t=1}^{T} P(y_t \mid y_1, \ldots, y_{t-1}, I)$$

Each factor is predicted by the decoder at timestep $t$, which attends to image features and all previously generated tokens.

#### Training Loss (Teacher-Forced Cross-Entropy)

During training, we maximize log-likelihood of the ground-truth caption:

$$\mathcal{L} = -\sum_{t=1}^{T} \log P(y_t \mid y_1, \ldots, y_{t-1}, I)$$

This is equivalent to standard cross-entropy at each position: feed the **ground-truth prefix** $(y_1, \ldots, y_{t-1})$ and predict $y_t$.

#### Teacher Forcing

At training step $t$, the decoder input is the **true** previous token $y_{t-1}$, not the model's own prediction $\hat{y}_{t-1}$.

| | Training (teacher forcing) | Inference (autoregressive) |
|--|---------------------------|---------------------------|
| Input at step $t$ | Ground-truth $y_{t-1}$ | Model's own $\hat{y}_{t-1}$ |
| Parallelism | All $T$ steps in one forward pass | Sequential — one token at a time |
| Exposure bias | Model never sees its mistakes during training | Errors compound at inference |

Teacher forcing enables efficient parallel training but creates a **train/test mismatch** — the model learns to continue perfect prefixes but must recover from its own errors at generation time.

### Causal Masking

The text decoder generates captions **autoregressively** — token $y_t$ may only depend on past tokens $y_1, \ldots, y_{t-1}$ and the image $I$. It must **not** peek at future tokens during training (that would be cheating).

This is enforced with a **causal (triangular) attention mask** $M$:

$$M_{ij} = \begin{cases} 0 & \text{if } j \leq i \quad \text{(allowed — attend to past/present)} \\ -\infty & \text{if } j > i \quad \text{(blocked — mask future)} \end{cases}$$

The attention scores become $\text{softmax}(QK^T / \sqrt{d_k} + M)$, so future positions receive zero weight.

#### Visual Example (sequence length $T = 4$)

```
         j=0   j=1   j=2   j=3
i=0  [   0     -∞    -∞    -∞  ]   ← token 0 sees only itself
i=1  [   0      0    -∞    -∞  ]   ← token 1 sees 0, 1
i=2  [   0      0     0    -∞  ]   ← token 2 sees 0, 1, 2
i=3  [   0      0     0     0  ]   ← token 3 sees all past
```

In PyTorch: `nn.Transformer.generate_square_subsequent_mask(T)` creates this upper-triangular $-\infty$ mask automatically. Without it, the decoder could "predict" token $t$ while already knowing token $t+1$ from the input — training loss would be artificially low and generation would fail.

In [ ]:
# Visualize the captioning architecture
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('Image Captioning Architecture', fontsize=18, fontweight='bold', pad=20)

# Image side
draw_architecture_block(ax, 3, 7, 3.5, 0.8, 'Image', '#E74C3C')
draw_architecture_block(ax, 3, 5.5, 3.5, 0.8, 'Vision Encoder (ViT)', '#E74C3C')
draw_architecture_block(ax, 3, 4, 3.5, 0.8, 'Image Features\n[N_patches, D]', '#C0392B')

draw_arrow(ax, (3, 6.5), (3, 6.0))
draw_arrow(ax, (3, 5.0), (3, 4.5))

# Decoder side
draw_architecture_block(ax, 10, 7, 4, 0.8, 'Text Tokens (shifted)', '#3498DB')
draw_architecture_block(ax, 10, 5.5, 4, 0.8, 'Self-Attention (causal)', '#3498DB')
draw_architecture_block(ax, 10, 4, 4, 0.8, 'Cross-Attention\nQ=text, K/V=image', '#9B59B6')
draw_architecture_block(ax, 10, 2.5, 4, 0.8, 'FFN + Softmax', '#2ECC71')
draw_architecture_block(ax, 10, 1, 4, 0.8, 'Next Word Prediction', '#F39C12')

draw_arrow(ax, (10, 6.5), (10, 6.0))
draw_arrow(ax, (10, 5.0), (10, 4.5))
draw_arrow(ax, (10, 3.5), (10, 3.0))
draw_arrow(ax, (10, 2.0), (10, 1.5))

# Cross-attention connection
draw_arrow(ax, (4.8, 4.0), (7.8, 4.0), color='#9B59B6', lw=2.5)
ax.text(6.3, 4.4, 'K, V', fontsize=11, color='#9B59B6', fontweight='bold')

plt.tight_layout()
plt.savefig('../assets/captioning_architecture.png', dpi=150, bbox_inches='tight')
plt.show()

![BLIP Architecture — Li et al. (2022)](../assets/paper_figure_blip.png)

*Source: Li et al. (2022) — "BLIP: Bootstrapping Language-Image Pre-training for Unified Vision-Language Understanding and Generation" — [arXiv:2201.12086](https://arxiv.org/abs/2201.12086)*

![BLIP-2 Framework — Li et al. (2023)](../../assets/paper_figures/blip2_framework.png)
*Source: Li et al. (2023) — BLIP-2: Bootstrapping Language-Image Pre-training with Frozen Image Encoders and Large Language Models — [arXiv:2301.12597](https://arxiv.org/abs/2301.12597)*

In [ ]:
# Build a small captioning model from scratch

class CaptioningDecoder(nn.Module):
    """Autoregressive decoder with cross-attention to image features."""
    def __init__(self, vocab_size=1000, embed_dim=128, n_heads=4,
                 n_layers=3, max_len=32):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_len, embed_dim)
        
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=embed_dim * 4, batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(embed_dim, vocab_size)
        self.embed_dim = embed_dim

    def forward(self, tgt_ids, memory):
        """Forward pass.
        tgt_ids: [B, T] text token ids
        memory:  [B, N, D] image features from encoder
        """
        B, T = tgt_ids.shape
        pos = torch.arange(T, device=tgt_ids.device).unsqueeze(0).expand(B, -1)
        x = self.token_embed(tgt_ids) + self.pos_embed(pos)
        
        # Causal mask (prevent attending to future tokens)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        
        # Decode with cross-attention to image features
        output = self.decoder(x, memory, tgt_mask=causal_mask)
        logits = self.output_proj(output)  # [B, T, vocab_size]
        return logits

    @torch.no_grad()
    def generate(self, memory, max_len=20, bos_id=1, eos_id=2):
        """Autoregressive generation."""
        B = memory.shape[0]
        generated = torch.full((B, 1), bos_id, dtype=torch.long, device=memory.device)
        
        for _ in range(max_len):
            logits = self.forward(generated, memory)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)
            if (next_token == eos_id).all():
                break
        return generated


# Simple vision encoder (reuse from before)
class SimpleVisionEncoder(nn.Module):
    def __init__(self, img_size=32, patch_size=4, embed_dim=128, n_layers=2):
        super().__init__()
        n_patches = (img_size // patch_size) ** 2
        self.patch_embed = nn.Conv2d(3, embed_dim, patch_size, patch_size)
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches, embed_dim) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=4, dim_feedforward=embed_dim*4, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        x = x + self.pos_embed
        return self.norm(self.encoder(x))  # [B, N_patches, D]


class ImageCaptioningModel(nn.Module):
    def __init__(self, vocab_size=1000, embed_dim=128):
        super().__init__()
        self.encoder = SimpleVisionEncoder(embed_dim=embed_dim)
        self.decoder = CaptioningDecoder(vocab_size=vocab_size, embed_dim=embed_dim)

    def forward(self, images, caption_ids):
        memory = self.encoder(images)
        return self.decoder(caption_ids, memory)

    @torch.no_grad()
    def generate(self, images, **kwargs):
        memory = self.encoder(images)
        return self.decoder.generate(memory, **kwargs)


model = ImageCaptioningModel(vocab_size=500, embed_dim=128)
count_parameters(model)

# Test
imgs = torch.randn(2, 3, 32, 32)
caps = torch.randint(0, 500, (2, 10))
logits = model(imgs, caps)
print(f"\nInput: images {imgs.shape}, captions {caps.shape}")
print(f"Output logits: {logits.shape} (predict next token at each position)")

In [ ]:
# Visualize: Autoregressive generation process

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle('Autoregressive Caption Generation (step by step)', 
             fontsize=14, fontweight='bold')

steps = [
    ('[BOS]', '→ a'),
    ('[BOS] a', '→ cute'),
    ('[BOS] a cute', '→ cat'),
    ('[BOS] a cute cat', '→ sitting'),
    ('[BOS] a cute cat sitting', '→ [EOS]'),
]

for ax, (context, prediction) in zip(axes, steps):
    ax.axis('off')
    # Show image
    ax.imshow(np.random.rand(8, 8, 3) * 0.3 + 0.4, extent=[0, 1, 0.5, 1.2])
    
    # Show context and prediction
    ax.text(0.5, 0.35, context, ha='center', fontsize=8, 
            bbox=dict(boxstyle='round', facecolor='#3498DB', alpha=0.3))
    ax.text(0.5, 0.1, prediction, ha='center', fontsize=10, fontweight='bold',
            color='#E74C3C')
    ax.set_xlim(-0.2, 1.2)
    ax.set_ylim(-0.1, 1.3)

plt.tight_layout()
plt.savefig('../assets/autoregressive_generation.png', dpi=150, bbox_inches='tight')
plt.show()

### Example 1: Cross-Entropy Loss — Token-Level Numerical Walkthrough

Let's trace the captioning loss for a 4-token caption: "[BOS] a cute cat"

**Setup:** At each position, the model predicts a probability distribution over the vocabulary (size V=500). The loss is computed per-token.

| Position $t$ | Input token | Target (next token) | Model's P(target) | Loss $-\log P$ |
|---|---|---|---|---|
| 0 | [BOS] | "a" | 0.72 | 0.33 |
| 1 | "a" | "cute" | 0.45 | 0.80 |
| 2 | "cute" | "cat" | 0.88 | 0.13 |
| 3 | "cat" | [EOS] | 0.65 | 0.43 |

**Total loss:** $\mathcal{L} = \frac{1}{4}(0.33 + 0.80 + 0.13 + 0.43) = 0.42$

The model is **least confident** about "cute" (P=0.45) — this gets the highest per-token loss, producing the strongest gradient signal. The model will adjust weights to better predict "cute" following "a" in the context of this image.

In [ ]:
# ============================================================
#  Example 1: Token-Level Cross-Entropy — Numerical Trace
# ============================================================

print("=" * 65)
print("  CAPTIONING LOSS: Token-Level Numerical Trace")
print("=" * 65)

# Simulate model output for caption "[BOS] a cute cat [EOS]"
vocab_size = 500
caption_tokens = ["[BOS]", "a", "cute", "cat", "[EOS]"]

# Token IDs (simulated)
token_ids = {"[BOS]": 1, "a": 42, "cute": 178, "cat": 95, "[EOS]": 2, "[PAD]": 0}

# Simulate model logits at each position (controlled randomness)
torch.manual_seed(42)
seq_len = 4  # We predict 4 tokens: a, cute, cat, [EOS]

# Create logits where model is more confident about some tokens
logits = torch.randn(1, seq_len, vocab_size) * 2
# Boost correct token logits to simulate a partially-trained model
target_ids = [token_ids["a"], token_ids["cute"], token_ids["cat"], token_ids["[EOS]"]]
confidence = [3.0, 1.5, 4.0, 2.5]  # how confident the model is

for t in range(seq_len):
    logits[0, t, target_ids[t]] += confidence[t]

# Compute per-token loss
probs = torch.softmax(logits, dim=-1)
target_tensor = torch.tensor([target_ids])

print(f"\nCaption: {' → '.join(caption_tokens)}")
print(f"\nToken-level breakdown:")
print(f"{'Position':>8} | {'Input':>8} | {'Target':>8} | {'P(target)':>10} | {'Loss':>8} | {'Top-3 predictions'}")
print("-" * 85)

total_loss = 0
for t in range(seq_len):
    p_target = probs[0, t, target_ids[t]].item()
    loss_t = -np.log(p_target)
    total_loss += loss_t
    
    # Top-3 predicted tokens
    top3_probs, top3_ids = probs[0, t].topk(3)
    top3_str = ", ".join(f"id={idx.item()}({p:.3f})" for idx, p in zip(top3_ids, top3_probs))
    
    input_tok = caption_tokens[t]
    target_tok = caption_tokens[t + 1]
    
    print(f"  {t:>6} | {input_tok:>8} | {target_tok:>8} | {p_target:>10.4f} | {loss_t:>8.4f} | [{top3_str}]")

avg_loss = total_loss / seq_len
print(f"\n  Average loss: {avg_loss:.4f}")
print(f"  Perplexity: exp({avg_loss:.4f}) = {np.exp(avg_loss):.2f}")
print(f"  (Perplexity = how many tokens the model is confused about)")

# Verify with PyTorch
pt_loss = F.cross_entropy(logits.squeeze(0), target_tensor.squeeze(0))
print(f"\n  PyTorch cross_entropy: {pt_loss.item():.4f} (matches ✅)" if abs(pt_loss.item() - avg_loss) < 0.001 else f"\n  PyTorch: {pt_loss.item():.4f}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Per-token loss
ax = axes[0]
per_token_losses = [-np.log(probs[0, t, target_ids[t]].item()) for t in range(seq_len)]
per_token_probs = [probs[0, t, target_ids[t]].item() for t in range(seq_len)]
colors = ['#2ECC71' if l < 1.0 else '#F39C12' if l < 2.0 else '#E74C3C' for l in per_token_losses]
bars = ax.bar(range(seq_len), per_token_losses, color=colors, alpha=0.8)
ax.set_xticks(range(seq_len))
ax.set_xticklabels([f'{caption_tokens[t]}→{caption_tokens[t+1]}' for t in range(seq_len)], fontsize=9)
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Per-Token Loss\n(red = model is confused)', fontsize=12, fontweight='bold')
for bar, loss, prob in zip(bars, per_token_losses, per_token_probs):
    ax.text(bar.get_x() + bar.get_width()/2, loss + 0.05, 
            f'P={prob:.3f}\nL={loss:.3f}', ha='center', fontsize=9)

# Probability distribution for hardest token
ax = axes[1]
hardest_t = np.argmax(per_token_losses)
top10_probs, top10_ids = probs[0, hardest_t].topk(10)
ax.barh(range(10), top10_probs.numpy(), color='#3498DB', alpha=0.8)
ax.set_yticks(range(10))
ax.set_yticklabels([f'token_{idx.item()}' + (' ← target' if idx.item() == target_ids[hardest_t] else '') 
                    for idx in top10_ids], fontsize=9)
ax.set_xlabel('Probability')
ax.set_title(f'Hardest Position: "{caption_tokens[hardest_t]}" → "{caption_tokens[hardest_t+1]}"\n(model\'s top-10 predictions)', fontsize=12, fontweight='bold')
ax.invert_yaxis()

# Teacher forcing vs autoregressive comparison
ax = axes[2]
modes = ['Teacher\nForcing', 'Autoregressive']
loss_vals = [avg_loss, avg_loss * 1.3]  # AR is typically worse due to error accumulation
ax.bar(modes, loss_vals, color=['#2ECC71', '#E74C3C'], alpha=0.8)
ax.set_ylabel('Average Loss')
ax.set_title('Training vs Inference Loss Gap\n(exposure bias)', fontsize=12, fontweight='bold')
for i, v in enumerate(loss_vals):
    ax.text(i, v + 0.05, f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('../assets/captioning_loss_trace.png', dpi=150, bbox_inches='tight')
plt.show()

### Example 2: Beam Search vs Greedy — Implementation and Comparison

Let's implement beam search and compare it with greedy decoding:

**Greedy** picks the single best token at each step → can miss globally optimal sequences.
**Beam search** keeps the top-B candidates at each step → explores more of the sequence space.

```
Step 1:  "a" (0.4)     "the" (0.3)    "one" (0.2)     ← B=3 beams
Step 2:  "a cat" (0.35) "a dog" (0.32) "the cat" (0.28)
Step 3:  "a cat sitting" (0.30)  "a dog playing" (0.29)  "a cat sleeping" (0.27)
```

Greedy would pick "a" → "cat" → "sitting" → ... (locally best at each step)
Beam search might find "the cute cat" has higher TOTAL score despite "the" being less likely at step 1.

In [ ]:
# ============================================================
#  Example 2: Beam Search Implementation and Comparison
# ============================================================

print("=" * 65)
print("  BEAM SEARCH vs GREEDY DECODING")
print("=" * 65)

@torch.no_grad()
def beam_search_decode(model, image, beam_width=3, max_len=15, bos_id=1, eos_id=2):
    """Beam search decoding for image captioning."""
    memory = model.encoder(image.unsqueeze(0))  # [1, N, D]
    
    # Initialize beams: (sequence, cumulative log-prob)
    beams = [(torch.tensor([[bos_id]]), 0.0)]
    completed = []
    
    for step in range(max_len):
        all_candidates = []
        
        for seq, score in beams:
            if seq[0, -1].item() == eos_id:
                completed.append((seq, score))
                continue
            
            logits = model.decoder(seq.to(memory.device), memory)
            log_probs = F.log_softmax(logits[0, -1], dim=-1)  # [vocab_size]
            
            # Get top-k next tokens
            topk_log_probs, topk_ids = log_probs.topk(beam_width)
            
            for i in range(beam_width):
                new_seq = torch.cat([seq, topk_ids[i:i+1].unsqueeze(0).cpu()], dim=1)
                new_score = score + topk_log_probs[i].item()
                all_candidates.append((new_seq, new_score))
        
        if not all_candidates:
            break
            
        # Keep top beam_width candidates
        all_candidates.sort(key=lambda x: x[1], reverse=True)
        beams = all_candidates[:beam_width]
    
    # Add remaining beams to completed
    completed.extend(beams)
    
    # Length-normalized scoring
    scored = [(seq, score / (seq.shape[1] ** 0.6)) for seq, score in completed]
    scored.sort(key=lambda x: x[1], reverse=True)
    
    return scored

# Run beam search vs greedy on a test image
test_img = torch.randn(3, 32, 32)

# Greedy decoding
model.eval()
greedy_output = model.generate(test_img.unsqueeze(0))
greedy_tokens = greedy_output[0].tolist()

# Beam search with different beam widths
print(f"\nDecoding comparison for same image:")
print(f"{'Method':<20} | {'Sequence (token IDs)':>40} | {'Score':>8}")
print("-" * 75)

print(f"{'Greedy (B=1)':<20} | {str(greedy_tokens[:12]):>40} | {'N/A':>8}")

for beam_width in [2, 3, 5]:
    results = beam_search_decode(model, test_img, beam_width=beam_width)
    best_seq, best_score = results[0]
    tokens = best_seq[0].tolist()[:12]
    print(f"{'Beam (B=' + str(beam_width) + ')':<20} | {str(tokens):>40} | {best_score:>8.3f}")

# Show all beam candidates for B=3
print(f"\nAll beam candidates (B=3):")
results = beam_search_decode(model, test_img, beam_width=3)
for rank, (seq, score) in enumerate(results[:5]):
    tokens = seq[0].tolist()
    print(f"  Rank {rank+1}: score={score:.3f}, tokens={tokens[:10]}...")

# Visualize decoding strategies
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Greedy vs Beam score distribution
ax = axes[0]
strategies = ['Greedy\n(B=1)', 'Beam\n(B=2)', 'Beam\n(B=3)', 'Beam\n(B=5)']
scores = []
for bw in [1, 2, 3, 5]:
    if bw == 1:
        scores.append(-2.5)  # approximate greedy score
    else:
        res = beam_search_decode(model, test_img, beam_width=bw)
        scores.append(res[0][1])

ax.bar(strategies, scores, color=['#E74C3C', '#F39C12', '#3498DB', '#2ECC71'], alpha=0.8)
ax.set_ylabel('Normalized Log-Probability')
ax.set_title('Decoding Quality vs Beam Width\n(higher = better sequence)', fontsize=12, fontweight='bold')

# Top-k and nucleus sampling illustration
ax = axes[1]
# Simulate a probability distribution
probs_sim = torch.softmax(torch.randn(20) * 2, dim=0).sort(descending=True).values.numpy()
x = range(len(probs_sim))
cumsum = np.cumsum(probs_sim)

ax.bar(x, probs_sim, color='#3498DB', alpha=0.4, label='Full distribution')
ax.bar(x[:5], probs_sim[:5], color='#E74C3C', alpha=0.8, label='Top-k (k=5)')
nucleus_k = np.searchsorted(cumsum, 0.9) + 1
ax.bar(x[:nucleus_k], probs_sim[:nucleus_k], color='#2ECC71', alpha=0.5, label=f'Nucleus (p=0.9, k={nucleus_k})')
ax.axhline(y=probs_sim[4], color='#E74C3C', linestyle='--', alpha=0.5)
ax.set_xlabel('Token rank')
ax.set_ylabel('Probability')
ax.set_title('Top-k vs Nucleus Sampling\n(nucleus adapts to distribution shape)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../assets/beam_search_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### Decoding Strategies

At inference, the decoder must choose how to select each next token. The choice trades off **quality**, **diversity**, and **speed**.

#### 1. Greedy Decoding

Always pick the highest-probability token:

$$y_t = \arg\max P(y_t \mid y_{<t}, I)$$

**Pros:** Fast (one forward pass per token). **Cons:** Can get stuck in repetitive or suboptimal sequences — the locally best choice is not always globally best.

#### 2. Beam Search

Maintain the top-$B$ partial sequences at each step. Score each candidate with **length normalization** to avoid favoring shorter captions:

$$\text{score} = \frac{1}{T^\alpha} \sum_{t=1}^{T} \log P(y_t \mid y_{<t}, I)$$

Typically $\alpha = 0.6$. Wider beam ($B = 5$–$10$) improves quality but increases compute linearly.

#### 3. Top-$k$ Sampling

Instead of always picking the argmax, **sample** from the top-$k$ most likely tokens. Renormalize their probabilities:

$$P'(y) = \frac{P(y) \cdot \mathbb{1}[y \in \text{top-}k]}{\sum_{y' \in \text{top-}k} P(y')}$$

Adds diversity — useful for creative captioning. $k = 50$ is a common default.

#### 4. Nucleus (Top-$p$) Sampling

Dynamically select the **smallest set of tokens** whose cumulative probability exceeds threshold $p$ (e.g., $p = 0.9$):

$$\text{nucleus} = \min \left\{ \mathcal{V}' : \sum_{y \in \mathcal{V}'} P(y) \geq p \right\}$$

Adapts $k$ per step — narrow when the model is confident, wide when uncertain. Often produces more natural text than fixed top-$k$.

## 2. Using Pretrained Models (BLIP)

In practice, you use pretrained models. BLIP is lightweight and works well.

In [ ]:
# Uncomment to run with pretrained BLIP (requires ~1GB download)
# from transformers import BlipProcessor, BlipForConditionalGeneration
# from PIL import Image
# import requests
#
# processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
# model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
#
# url = "https://images.unsplash.com/photo-1574158622682-e40e69881006"
# image = Image.open(requests.get(url, stream=True).raw)
#
# inputs = processor(image, return_tensors="pt")
# output = model.generate(**inputs, max_length=50)
# caption = processor.decode(output[0], skip_special_tokens=True)
# print(f"Caption: {caption}")

print("To run pretrained BLIP, uncomment the cell above.")
print("It needs ~1GB download but works great on CPU!")
print("\nBlip models available:")
print("  - Salesforce/blip-image-captioning-base   (~990MB, good for CPU)")
print("  - Salesforce/blip-image-captioning-large  (~1.8GB, better quality)")

### Evaluation Metrics

Image captioning is evaluated by comparing generated captions to human reference captions. No single metric captures everything — each emphasizes different aspects of quality.

#### BLEU (Bilingual Evaluation Understudy)

Measures **n-gram precision** with a brevity penalty to discourage overly short captions:

$$\text{BLEU} = BP \cdot \exp\left(\sum_{n=1}^{4} w_n \log p_n\right)$$

where $p_n$ is the precision of n-grams of order $n$ (typically $w_n = 1/4$), and the brevity penalty is:

$$BP = \min\left(1,\; e^{1 - r/c}\right)$$

with $r$ = reference length and $c$ = candidate length. High BLEU means the generated caption shares many word sequences with references — but it penalizes valid paraphrases.

#### CIDEr (Consensus-Based Image Description Evaluation)

Uses **TF-IDF weighted n-gram similarity**, rewarding words that are both frequent in references *and* distinctive across the dataset. CIDEr correlates well with human judgment because it favors unique, descriptive language ("golden retriever" > "dog").

#### METEOR (Metric for Evaluation of Translation with Explicit ORdering)

Goes beyond exact n-gram matching: incorporates **synonyms, stemming, and word order**. More recall-friendly than BLEU — "automobile" matches "car".

#### Metric Comparison

| Metric | Focus | Strength | Weakness |
|--------|-------|----------|----------|
| **BLEU** | N-gram precision | Fast, widely used | Ignores synonyms; favors short, safe captions |
| **CIDEr** | Consensus / descriptiveness | Best human correlation on MSCOCO | Sensitive to dataset-specific vocabulary |
| **METEOR** | Recall + semantics | Handles paraphrases | Slower; still misses deep semantics |

In practice, report **multiple metrics** — a model can score high on BLEU while producing bland captions that humans dislike.

In [ ]:
# ============================================================
#  Example 3: BLEU Score — Hand Computation
# ============================================================

print("=" * 65)
print("  BLEU SCORE: Step-by-Step Computation")
print("=" * 65)

def compute_bleu_manual(candidate, reference, max_n=4):
    """Compute BLEU score step by step."""
    cand_tokens = candidate.lower().split()
    ref_tokens = reference.lower().split()
    
    print(f"\n  Candidate: '{candidate}'")
    print(f"  Reference: '{reference}'")
    print(f"  Candidate length (c): {len(cand_tokens)}")
    print(f"  Reference length (r): {len(ref_tokens)}")
    
    # Brevity penalty
    if len(cand_tokens) < len(ref_tokens):
        bp = np.exp(1 - len(ref_tokens) / len(cand_tokens))
    else:
        bp = 1.0
    print(f"\n  Brevity Penalty: BP = min(1, exp(1 - {len(ref_tokens)}/{len(cand_tokens)})) = {bp:.4f}")
    
    # N-gram precision for each n
    precisions = []
    print(f"\n  N-gram Precision:")
    for n in range(1, max_n + 1):
        # Generate n-grams
        cand_ngrams = [' '.join(cand_tokens[i:i+n]) for i in range(len(cand_tokens) - n + 1)]
        ref_ngrams = [' '.join(ref_tokens[i:i+n]) for i in range(len(ref_tokens) - n + 1)]
        
        if not cand_ngrams:
            precisions.append(0)
            print(f"    {n}-gram: 0/{0} = 0.000 (no {n}-grams in candidate)")
            continue
        
        # Clipped count
        matches = 0
        ref_counts = {}
        for ng in ref_ngrams:
            ref_counts[ng] = ref_counts.get(ng, 0) + 1
        
        for ng in cand_ngrams:
            if ng in ref_counts and ref_counts[ng] > 0:
                matches += 1
                ref_counts[ng] -= 1
        
        precision = matches / len(cand_ngrams) if cand_ngrams else 0
        precisions.append(precision)
        
        # Show n-grams
        matched_str = [ng for ng in cand_ngrams if ng in set(ref_ngrams)]
        print(f"    {n}-gram: {matches}/{len(cand_ngrams)} = {precision:.3f}  "
              f"matched: {matched_str[:5]}")
    
    # BLEU = BP * exp(mean of log precisions)
    log_precisions = [np.log(max(p, 1e-10)) for p in precisions]
    bleu = bp * np.exp(np.mean(log_precisions))
    
    print(f"\n  BLEU-4 = BP × exp(mean(log(p_1..p_4)))")
    print(f"         = {bp:.4f} × exp(mean({[f'{lp:.3f}' for lp in log_precisions]}))")
    print(f"         = {bp:.4f} × exp({np.mean(log_precisions):.4f})")
    print(f"         = {bleu:.4f}")
    
    return bleu

# Example 1: Good caption
print("\n" + "=" * 65)
print("  Case 1: High-quality caption")
bleu1 = compute_bleu_manual(
    "a cat sitting on a couch",
    "a cat is sitting on the couch"
)

# Example 2: Mediocre caption
print("\n" + "=" * 65)
print("  Case 2: Partially correct caption")
bleu2 = compute_bleu_manual(
    "the animal on furniture",
    "a cat is sitting on the couch"
)

# Example 3: Bad caption
print("\n" + "=" * 65)
print("  Case 3: Wrong caption")
bleu3 = compute_bleu_manual(
    "a dog running in a park",
    "a cat is sitting on the couch"
)

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
captions = ['Good:\n"a cat sitting\non a couch"', 'Mediocre:\n"the animal\non furniture"', 'Bad:\n"a dog running\nin a park"']
scores = [bleu1, bleu2, bleu3]
colors = ['#2ECC71', '#F39C12', '#E74C3C']
bars = ax.bar(captions, scores, color=colors, alpha=0.8)
ax.set_ylabel('BLEU-4 Score')
ax.set_title('BLEU Score Comparison\n(Reference: "a cat is sitting on the couch")', fontsize=13, fontweight='bold')
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, score + 0.01, f'{score:.4f}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, max(max(scores) * 1.3, 0.1))

plt.tight_layout()
plt.savefig('../assets/bleu_score_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Takeaways

1. **Image captioning = Vision Encoder + Text Decoder with Cross-Attention**
2. The decoder generates words **autoregressively** (one at a time)
3. **Cross-attention** lets each generated word look at relevant image regions
4. **Teacher forcing** during training: feed ground-truth tokens, predict next
5. For practical use: **BLIP** is lightweight and works on CPU

---
**Next:** `03_visual_question_answering.ipynb` - Answer questions about images

---

## 📚 References & Further Reading

### Papers
- **Show, Attend and Tell: Neural Image Caption Generation with Visual Attention** — Xu et al., 2015 — [arXiv:1502.03044](https://arxiv.org/abs/1502.03044) — Pioneering attention-based captioning
- **BLIP: Bootstrapping Language-Image Pre-training** — Li et al., 2022 — [arXiv:2201.12086](https://arxiv.org/abs/2201.12086) — Unified vision-language pretraining
- **BLIP-2: Bootstrapping Language-Image Pre-training with Frozen Image Encoders and LLMs** — Li et al., 2023 — [arXiv:2301.12597](https://arxiv.org/abs/2301.12597) — Efficient captioning with Q-Former
- **CoCa: Contrastive Captioners** — Yu et al., 2022 — [arXiv:2205.01917](https://arxiv.org/abs/2205.01917) — Dual encoder-decoder architecture
- **GIT: A Generative Image-to-text Transformer** — Wang et al., 2022 — [arXiv:2205.14100](https://arxiv.org/abs/2205.14100) — Simple and effective captioning
- **BLEU: a Method for Automatic Evaluation of Machine Translation** — Papineni et al., 2002 — [ACL Anthology](https://aclanthology.org/P02-1040/) — BLEU metric paper
- **CIDEr: Consensus-based Image Description Evaluation** — Vedantam et al., 2015 — [arXiv:1411.5726](https://arxiv.org/abs/1411.5726) — CIDEr metric paper

### Blog Posts & Cheat Sheets
- 🔗 [The Illustrated Image Captioning](https://jalammar.github.io/illustrated-image-captioning/) — Jay Alammar — Visual captioning walkthrough
- 🔗 [Hugging Face BLIP Guide](https://huggingface.co/docs/transformers/model_doc/blip) — Use BLIP for captioning in 5 lines
- 🔗 [Karpathy's NeuralTalk](https://cs.stanford.edu/people/karpathy/deepimagesent/) — Classic captioning demo and dataset splits
- 🔗 [Decoding Strategies in NLG](https://huggingface.co/blog/how-to-generate) — Hugging Face — Beam search, top-k, nucleus sampling explained
- 🔗 [MS COCO Captions](https://cocodataset.org/#captions-2015) — Standard captioning benchmark dataset
- 🔗 [Papers With Code: Image Captioning](https://paperswithcode.com/task/image-captioning) — Latest SOTA models and benchmarks